# 자체 서명 인증서: AgentCore Gateway를 위한 ALB 솔루션

이 실습에서는 [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)를 **자체 서명 TLS 인증서**를 사용하는 API에 연결하는 방법을 알아봅니다.

## 문제

AgentCore Gateway는 **공개 루트 CA만**을 기준으로 TLS 인증서를 검증합니다. API가 자체 서명 인증서를 사용하면 AgentCore Gateway는 해당 API와 TLS 연결을 설정할 수 없습니다. 이는 API가 ALB 또는 NLB 뒤에 있거나 EC2 인스턴스, ECS/EKS 또는 다른 컴퓨팅 플랫폼에서 실행되는지와 관계없이 적용됩니다.

일반적인 해결 방법은 자체 서명 인증서를 공개 ACM 인증서로 교체하는 것입니다. 하지만 **공개 ACM 인증서는 도메인 소유권을 증명하기 위한 DNS 검증이 필요합니다**. 인증서의 도메인을 소유하고 있지 않다면(예: 다른 팀에서 관리하는 내부 도메인) 해당 도메인에 대한 공개 인증서를 발급받을 수 없습니다.


![아키텍처](./images/self-signed.png)

## 해결 방법

**가장 간단한 해결 방법:** 가능하다면 자체 서명 인증서를 소유한 도메인의 공개 ACM 인증서로 교체합니다. 이 방법은 새로운 인프라를 추가하지 않아도 됩니다.

**이 방법을 사용할 수 없다면**(예: 도메인을 소유하지 않은 경우) API 앞에 **공개 인증서가 연결된 내부 ALB**를 배포합니다.

1. 소유한 도메인에 대한 공개 ACM 인증서를 발급받습니다(예: `*.external.yourcompany.com`).
2. TLS 종료를 위해 공개 인증서가 연결된 새 내부 ALB를 배포합니다.
3. 새 ALB가 API를 대상으로 지정하도록 설정합니다.
4. AgentCore Gateway 대상에서 `routingDomain`을 사용하여 공개적으로 확인 가능한 새 ALB의 DNS를 통해 라우팅합니다.

새 ALB의 도메인은 API의 도메인과 일치할 **필요가 없습니다**. 새 ALB는 공개 인증서로 TLS를 종료한 다음 HTTPS를 통해 API로 전달합니다. 내부 사용자를 위한 기존 구성은 변경되지 않습니다.

## 아키텍처


**단계별 작동 방식:**

1. **대상 URL**을 공개 ACM 인증서와 일치하는 도메인으로 설정합니다(예: `https://my-server.my-company.com`).
2. **`routingDomain`**을 내부 ALB DNS 이름으로 설정합니다(예: `internal-my-alb-1234567890.us-west-2.elb.amazonaws.com`).
3. VPC Lattice는 라우팅 도메인을 통해 트래픽을 ALB로 라우팅합니다. TLS SNI는 ALB의 공개 ACM 인증서와 일치하는 `my-server.my-company.com`으로 설정되므로 **TLS 핸드셰이크가 성공합니다**.
4. ALB는 **TLS를 종료**하고 **호스트 헤더 변환**을 적용하여 `Host` 헤더를 `my-server.my-company.com`에서 프라이빗 리소스의 도메인(예: `my-server.my-company.internal`)으로 다시 작성합니다.
5. ALB는 자체 서명 인증서를 사용하여 **HTTPS를 통해 백엔드로 요청을 전달합니다**. 모든 트래픽은 VPC 내부에 유지됩니다.

VPC 송신 및 관리형 VPC 리소스에 대한 배경 정보는 [프로젝트 README](../README.md)와 [고급 개념 README](./README.md)를 참조하세요.

## 사전 요구 사항

- [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb) 완료(VPC + AgentCore Gateway 배포)
- 소유한 도메인에 대한 [ACM 공개 인증서](../00-prerequisites/create-acm-public-certificate.md)

## 1단계: 종속성 설치 및 라이브러리 가져오기

In [ ]:
import os
from pathlib import Path

# 프로젝트 루트로 이동
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

!pip install --force-reinstall -q -r requirements.txt

In [ ]:
import json
import os
import time

import boto3
from utils.utils import get_token

# 실습 0에서 변수 복원
%store -r ACCOUNT_A_ID
%store -r ACCOUNT_A_PROFILE
%store -r GATEWAY_ID
%store -r GATEWAY_URL
%store -r USER_POOL_ID
%store -r USER_POOL_CLIENT_ID
%store -r TOKEN_ENDPOINT_URL
%store -r OAUTH_SCOPES
%store -r VPC_USW2_ID
%store -r VPC_USW2_PRIVATE_SUBNETS

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID

REGION = "us-west-2"
session = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION)
agentcore = session.client("bedrock-agentcore-control")

# Cognito 클라이언트 보안 암호 가져오기
cognito = session.client("cognito-idp")
client_desc = cognito.describe_user_pool_client(UserPoolId=USER_POOL_ID, ClientId=USER_POOL_CLIENT_ID)
CLIENT_SECRET = client_desc["UserPoolClient"]["ClientSecret"]

print(f"Account:    {ACCOUNT_A_ID}")
print(f"Region:     {REGION}")
print(f"Gateway ID: {GATEWAY_ID}")
print(f"VPC ID:     {VPC_USW2_ID}")

## 2단계: 자체 서명 인증서를 사용하는 샘플 백엔드 배포

이 실습에서는 자체 서명 TLS 인증서를 사용하는 샘플 API를 배포합니다. 이는 자체 서명 인증서를 사용하는 내부 엔드포인트를 시뮬레이션하며, API가 실행되는 환경과 관계없이 동일한 해결 방법을 적용할 수 있습니다.

**SelfSignedBackend**는 다음 리소스를 배포합니다.
   - 사용자 데이터에서 `openssl`을 통해 생성한 자체 서명 인증서를 사용하여 HTTPS 포트 443에서 REST API(FastAPI)를 실행하는 **EC2 인스턴스**
   - `api.internal.<baseDomain>`이 EC2의 프라이빗 IP를 가리키는 A 레코드를 포함한 **Route 53 프라이빗 호스팅 영역**(`internal.<baseDomain>`)
   - EC2에 공개적으로 확인 가능한 DNS 이름을 제공하는 **내부 NLB**(포트 443의 TCP 패스스루)

구성은 다음과 같습니다.

```
Private hosted zone:  api.internal.<baseDomain> → EC2 private IP
NLB (TCP passthrough): internal-xxx.elb.amazonaws.com:443 → EC2:443
EC2:                  HTTPS:443 with self-signed cert (CN=api.internal.<baseDomain>)
```

**프라이빗 호스팅 영역**은 VPC 내에서 `api.internal.<baseDomain>`을 인증서의 CN과 일치하는 EC2 프라이빗 IP로 확인합니다.

프라이빗 호스팅 영역의 도메인은 공개적으로 확인할 수 없으므로 **NLB**는 VPC Lattice가 `routingDomain`으로 사용할 수 있는 공개적으로 확인 가능한 DNS 이름(`internal-xxx.elb.amazonaws.com`)을 제공합니다. NLB는 TCP 패스스루를 수행하고 TLS를 종료하지 않으므로 EC2의 자체 서명 인증서가 클라이언트에 직접 제시됩니다.

> **참고:** 자체 서명 인증서 대신 AWS Private CA에서 발급한 인증서가 있는 경우에도 동일한 해결 방법이 적용됩니다. [프라이빗 인증 기관 실습](./02-private-certificate-authority.ipynb)을 참조하세요.

In [ ]:
!cdk deploy SelfSignedBackend --profile {ACCOUNT_A_PROFILE} --require-approval never --outputs-file backend-outputs.json

In [ ]:
with open("backend-outputs.json") as f:
    backend_outputs = json.load(f)["SelfSignedBackend"]

API_KEY_VALUE = backend_outputs["ApiKey"]
EC2_INSTANCE_ID = backend_outputs["Ec2InstanceId"]
EC2_PRIVATE_IP = backend_outputs["Ec2PrivateIp"]
CERT_ARN_SELF = backend_outputs["CertArn"]
CERT_DOMAIN = backend_outputs["CertDomain"]
NLB_DNS = backend_outputs["NlbDnsName"]
NLB_SG_ID = backend_outputs["NlbSgId"]

print(f"EC2 instance:         {EC2_INSTANCE_ID}")
print(f"EC2 private IP:       {EC2_PRIVATE_IP}")
print(f"Self-signed cert:     {CERT_ARN_SELF}")
print(f"Certificate domain:   {CERT_DOMAIN}")
print(f"NLB DNS:              {NLB_DNS}")
print(f"NLB SG:               {NLB_SG_ID}")
print(f"EC2 HTTPS endpoint:   https://{CERT_DOMAIN}:443")

In [ ]:
# API 키 자격 증명 공급자 생성(선택적 실패 테스트와 정상 작동 대상 모두에 필요)
cred_response = agentcore.create_api_key_credential_provider(
    name="self-signed-proxy-api-key",
    apiKey=API_KEY_VALUE,
)
CRED_PROVIDER_ARN = cred_response["credentialProviderArn"]
print(f"Credential provider ARN: {CRED_PROVIDER_ARN}")

### AgentCore Gateway에서 작동하지 않는 이유(선택 사항)

API는 **자체 서명 인증서**를 사용합니다. 이 엔드포인트를 가리키는 AgentCore Gateway 대상을 생성하면 대상 생성은 **성공**하고 대상 상태는 `READY`가 됩니다. 하지만 도구를 **호출**하면 AgentCore Gateway가 **공개 루트 CA만**을 기준으로 인증서를 검증하므로 TLS 핸드셰이크가 실패합니다.

호출 시 다음 오류가 표시됩니다.

```bash
{
  "jsonrpc": "2.0",
  "id": 2,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "OpenAPIClientException - Error executing HTTP request for unknown: (certificate_unknown) PKIX path building failed: sun.security.provider.certpath.SunCertPathBuilderException: unable to find valid certification path to requested target"
      }
    ],
    "isError": true
  }
}
```

이를 확인하려면 아래 셀의 주석을 해제하고 실행하세요. 이 셀들은 대상을 생성하고 준비될 때까지 기다린 다음, 대상을 호출하여 TLS 오류를 확인하고 리소스를 정리합니다. 이 작업에는 몇 분이 걸립니다. **바로 해결 방법으로 이동하려면 이 셀들을 건너뛰세요.**

In [ ]:
# # EC2 인스턴스는 자체 서명 인증서로 HTTPS:443을 제공합니다.
# # NLB는 VPC Lattice가 라우팅할 수 있도록 공개적으로 확인 가능한 DNS(routingDomain)를
# # 제공합니다. 하지만 자체 서명 인증서로 인해 TLS 오류가 발생합니다.
# with open("03-advanced-concepts/openapi-private.json") as f:
#     fail_schema = json.load(f)

# FAIL_ENDPOINT = f"https://{CERT_DOMAIN}"
# fail_schema["servers"] = [{"url": FAIL_ENDPOINT}]

# print(f"Target URL: {FAIL_ENDPOINT}")
# print(f"routingDomain: {NLB_DNS}")
# print(f"This will fail because the self-signed cert is not trusted by AgentCore.")

In [ ]:
# # NLB를 routingDomain으로 사용하여 대상 생성 - 호출 시 TLS 실패
# fail_response = agentcore.create_gateway_target(
#     gatewayIdentifier=GATEWAY_ID,
#     name="self-signed-test",
#     description="Testing self-signed cert — expected to fail on invocation",
#     targetConfiguration={
#         "mcp": {
#             "openApiSchema": {
#                 "inlinePayload": json.dumps(fail_schema),
#             }
#         }
#     },
#     credentialProviderConfigurations=[
#         {
#             "credentialProviderType": "API_KEY",
#             "credentialProvider": {
#                 "apiKeyCredentialProvider": {
#                     "providerArn": CRED_PROVIDER_ARN,
#                     "credentialParameterName": "x-api-key",
#                     "credentialLocation": "HEADER",
#                 }
#             },
#         }
#     ],
#     privateEndpoint={
#         "managedVpcResource": {
#             "vpcIdentifier": VPC_USW2_ID,
#             "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
#             "endpointIpAddressType": "IPV4",
#             "securityGroupIds": [NLB_SG_ID],
#             "routingDomain": NLB_DNS,
#         }
#     },
# )

# FAIL_TARGET_ID = fail_response["targetId"]
# print(f"Target ID: {FAIL_TARGET_ID}")
# print(f"Status:    {fail_response['status']}")

In [ ]:
# # 대상이 준비될 때까지 대기
# while True:
#     fail_target = agentcore.get_gateway_target(
#         gatewayIdentifier=GATEWAY_ID, targetId=FAIL_TARGET_ID
#     )
#     status = fail_target["status"]
#     print(f"Status: {status}")
#     if status == "READY":
#         print("\nTarget is ready — but invocation will fail due to self-signed cert.")
#         break
#     if status == "FAILED":
#         print(f"\nTarget creation failed: {fail_target.get('statusReasons', [])}")
#         break
#     time.sleep(30)

In [ ]:
# # 대상 호출 - TLS/인증서 오류 예상
# import requests
#
# token_response = get_token(
#     token_endpoint_url=TOKEN_ENDPOINT_URL,
#     client_id=USER_POOL_CLIENT_ID,
#     client_secret=CLIENT_SECRET,
#     scope_string=OAUTH_SCOPES.replace(",", " "),
# )
#
# headers = {
#     "Authorization": f"Bearer {token_response['access_token']}",
#     "Content-Type": "application/json",
# }
#
# response = requests.post(
#     GATEWAY_URL,
#     headers=headers,
#     json={
#         "jsonrpc": "2.0",
#         "method": "tools/call",
#         "params": {"name": "self-signed-test___healthCheck", "arguments": {}},
#         "id": 2,
#     },
# )
# print("Health check:")
# print(json.dumps(response.json(), indent=2))

In [ ]:
# # 실패한 대상 정리
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=FAIL_TARGET_ID)
# print(f"Deleting failed target: {FAIL_TARGET_ID}")
# while True:
#     try:
#         t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=FAIL_TARGET_ID)
#         print(f"  Status: {t['status']}")
#         time.sleep(15)
#     except agentcore.exceptions.ResourceNotFoundException:
#         print("  Target deleted.")
#         break

### 옵션

| 옵션 | 사용 시점 |
|--------|-------------|
| 기존 엔드포인트의 **자체 서명 인증서를 공개 인증서로 교체** | 도메인을 소유하고 있고 해당 도메인에 대한 공개 ACM 인증서를 발급받을 수 있는 경우의 가장 간단한 해결 방법 |
| 앞단에 **공개 인증서가 연결된 새 ALB 추가**(이 실습) | 기존 엔드포인트의 인증서를 수정할 수 없거나 도메인을 소유하지 않은 경우 |

옵션 2로 진행하여 API 앞에 공개 인증서가 연결된 새 내부 ALB를 배포하겠습니다.

## 3단계: 공개 인증서 구성

**소유한** 도메인에 대한 ACM 공개 인증서 ARN을 입력합니다. 어떤 도메인이든 사용할 수 있으며 백엔드 API의 도메인과 일치할 필요는 없습니다. 예를 들어 인증서가 `*.external.yourcompany.com`에 대한 것이라면 `api.external.yourcompany.com`과 같은 하위 도메인을 입력합니다.

이 도메인은 AgentCore Gateway 대상 URL로 사용됩니다. 이 도메인에 대한 공개 DNS 레코드는 **필요하지 않습니다**. `routingDomain`이 새 ALB의 DNS를 통한 라우팅을 처리합니다.

In [ ]:
CERT_ARN = input("ACM public certificate ARN: ")
DOMAIN = input("Domain name covered by the certificate (e.g., api.external.yourcompany.com): ")

assert CERT_ARN.startswith("arn:aws:acm:"), "Invalid certificate ARN"
assert not DOMAIN.startswith("http"), "Domain should not include http:// or https://"
assert "." in DOMAIN, "Domain must contain at least one dot"

print(f"Cert ARN: {CERT_ARN}")
print(f"Domain:   {DOMAIN}")

## 4단계: 공개 인증서 프록시 ALB 생성

이 해결 방법은 API 앞에 공개 ACM 인증서가 연결된 **새 내부 ALB**를 배포합니다. 전달해야 할 파라미터를 정확히 확인할 수 있도록 boto3를 사용하여 생성합니다.

다음 하위 단계는 [설명서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/vpc-egress-private-endpoints.html)의 절차와 동일합니다.

1. 동일한 VPC와 프라이빗 서브넷에 **내부 ALB를 생성**합니다.
2. **HTTPS 포트 443**에서 API의 IP 주소를 가리키는 **IP 기반 대상 그룹을 생성**합니다.
3. 공개 ACM 인증서와 `Host` 헤더를 공개 도메인에서 프라이빗 도메인으로 다시 작성하는 **호스트 헤더 변환**을 사용하는 **HTTPS 리스너를 생성**합니다.

트래픽 흐름은 [설명서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/vpc-egress-private-endpoints.html)의 내용과 같습니다.

1. VPC Lattice가 `routingDomain`을 통해 트래픽을 프록시 ALB로 라우팅합니다.
2. TLS SNI가 공개 ACM 인증서와 일치하여 TLS 핸드셰이크가 성공합니다.
3. ALB가 **TLS를 종료**하고 **호스트 헤더 변환**(`DOMAIN` → `CERT_DOMAIN`)을 적용합니다.
4. ALB가 자체 서명 인증서를 사용하여 **HTTPS를 통해 백엔드로 요청을 전달**합니다.
5. 모든 트래픽은 VPC 내부에 유지됩니다.

> **참고:** ALB는 HTTPS를 통해 전달할 때 백엔드의 자체 서명 인증서를 검증하지 않습니다. 이는 표준 ALB 동작이며 백엔드 인증서 검증은 수행되지 않습니다.

In [ ]:
elbv2 = session.client("elbv2")
ec2_client = session.client("ec2")

# 보안 그룹 인바운드 규칙에 사용할 VPC CIDR 가져오기
vpc_info = ec2_client.describe_vpcs(VpcIds=[VPC_USW2_ID])
VPC_CIDR = vpc_info["Vpcs"][0]["CidrBlock"]

# 프록시 ALB용 보안 그룹 생성
sg_response = ec2_client.create_security_group(
    GroupName=f"proxy-alb-sg-{int(time.time())}",
    Description="Public cert proxy ALB - HTTPS from VPC",
    VpcId=VPC_USW2_ID,
)
PROXY_ALB_SG_ID = sg_response["GroupId"]

ec2_client.authorize_security_group_ingress(
    GroupId=PROXY_ALB_SG_ID,
    IpPermissions=[
        {
            "IpProtocol": "tcp",
            "FromPort": 443,
            "ToPort": 443,
            "IpRanges": [{"CidrIp": VPC_CIDR, "Description": "HTTPS from VPC"}],
        }
    ],
)
print(f"Security group: {PROXY_ALB_SG_ID}")

# 내부 Application Load Balancer 생성
alb_response = elbv2.create_load_balancer(
    Name="self-signed-proxy-alb",
    Subnets=VPC_USW2_PRIVATE_SUBNETS,
    SecurityGroups=[PROXY_ALB_SG_ID],
    Scheme="internal",
    Type="application",
    IpAddressType="ipv4",
)
PROXY_ALB_ARN = alb_response["LoadBalancers"][0]["LoadBalancerArn"]
PROXY_ALB_DNS = alb_response["LoadBalancers"][0]["DNSName"]
print(f"ALB ARN: {PROXY_ALB_ARN}")
print(f"ALB DNS: {PROXY_ALB_DNS}")

# ALB가 활성화될 때까지 대기
print("Waiting for ALB to become active...")
waiter = elbv2.get_waiter("load_balancer_available")
waiter.wait(LoadBalancerArns=[PROXY_ALB_ARN])
print("ALB is active.")

In [ ]:
# HTTPS:443에서 백엔드를 가리키는 IP 기반 대상 그룹 생성
# EC2 인스턴스는 자체 서명 인증서를 사용하여 HTTPS 제공
tg_response = elbv2.create_target_group(
    Name="self-signed-proxy-tg",
    Protocol="HTTPS",
    Port=443,
    VpcId=VPC_USW2_ID,
    TargetType="ip",
    HealthCheckProtocol="HTTPS",
    HealthCheckPort="443",
    HealthCheckPath="/health",
)
TARGET_GROUP_ARN = tg_response["TargetGroups"][0]["TargetGroupArn"]
print(f"Target group: {TARGET_GROUP_ARN}")

# HTTPS 포트 443에 백엔드의 프라이빗 IP 등록
elbv2.register_targets(
    TargetGroupArn=TARGET_GROUP_ARN,
    Targets=[{"Id": EC2_PRIVATE_IP, "Port": 443}],
)
print(f"Registered target: {EC2_PRIVATE_IP}:443 (HTTPS)")

In [ ]:
# 공개 ACM 인증서를 사용하는 HTTPS 리스너 생성
listener_response = elbv2.create_listener(
    LoadBalancerArn=PROXY_ALB_ARN,
    Protocol="HTTPS",
    Port=443,
    Certificates=[{"CertificateArn": CERT_ARN}],
    DefaultActions=[
        {
            "Type": "forward",
            "TargetGroupArn": TARGET_GROUP_ARN,
            "ForwardConfig": {"TargetGroups": [{"TargetGroupArn": TARGET_GROUP_ARN, "Weight": 1}]},
        }
    ],
)
LISTENER_ARN = listener_response["Listeners"][0]["ListenerArn"]
print(f"Listener ARN: {LISTENER_ARN}")

# 호스트 헤더 변환을 사용하는 규칙 추가(기본 규칙에는 변환을 사용할 수 없음)
# Host 헤더를 공개 도메인에서 프라이빗 도메인으로 다시 작성
elbv2.create_rule(
    ListenerArn=LISTENER_ARN,
    Priority=1,
    Conditions=[
        {
            "Field": "path-pattern",
            "PathPatternConfig": {"Values": ["/*"]},
        }
    ],
    Actions=[
        {
            "Type": "forward",
            "TargetGroupArn": TARGET_GROUP_ARN,
            "ForwardConfig": {"TargetGroups": [{"TargetGroupArn": TARGET_GROUP_ARN, "Weight": 1}]},
        }
    ],
    Transforms=[
        {
            "Type": "host-header-rewrite",
            "HostHeaderRewriteConfig": {"Rewrites": [{"Regex": ".*", "Replace": CERT_DOMAIN}]},
        }
    ],
)
print(f"Host header transform: {DOMAIN} -> {CERT_DOMAIN}")

In [ ]:
print("Proxy ALB created successfully!")
print(f"  ALB DNS (routingDomain): {PROXY_ALB_DNS}")
print(f"  Security Group ID:       {PROXY_ALB_SG_ID}")
print()
print("Traffic flow:")
print("  Proxy ALB (public cert :443) -> EC2 (self-signed cert :443)")
print(f"  routingDomain: {PROXY_ALB_DNS}")
print(f"  Backend IP:    {EC2_PRIVATE_IP}:443")

## 5단계: AgentCore Gateway 대상 생성

`routingDomain`이 포함된 [관리형 VPC 리소스](../01-managed-vpc-resource/README.md)를 사용하여 게이트웨이 대상을 생성합니다.

전체 트래픽 흐름은 다음과 같습니다.

1. **대상 URL**은 공개 ACM 인증서와 일치하는 도메인(`https://{DOMAIN}`)을 사용합니다.
2. **`routingDomain`**은 공개적으로 확인 가능한 프록시 ALB DNS 이름입니다.
3. VPC Lattice는 라우팅 도메인을 통해 트래픽을 ALB로 라우팅합니다. TLS SNI는 ALB의 공개 ACM 인증서와 일치하는 `{DOMAIN}`으로 설정되므로 **TLS 핸드셰이크가 성공합니다**.
4. ALB가 **TLS를 종료**하고 **호스트 헤더 변환**(`{DOMAIN}` → `{CERT_DOMAIN}`)을 적용합니다.
5. ALB가 자체 서명 인증서를 사용하여 **HTTPS를 통해 백엔드로 요청을 전달**합니다.
6. 모든 트래픽은 VPC 내부에 유지됩니다.

API의 자체 서명 인증서는 그대로 유지됩니다. 내부 사용자는 계속 API에 직접 액세스하고, AgentCore Gateway는 공개 인증서가 연결된 새 ALB를 사용합니다.

> **보안 그룹:** Resource Gateway ENI가 포트 443에서 새 ALB에 연결할 수 있도록 **프록시 ALB의** 보안 그룹을 `securityGroupIds`에 전달합니다.

In [ ]:
# OpenAPI 스키마를 로드하고 서버 URL을 공개 도메인으로 설정
with open("03-advanced-concepts/openapi-private.json") as f:
    openapi_schema = json.load(f)

TARGET_ENDPOINT = f"https://{DOMAIN}"
openapi_schema["servers"] = [{"url": TARGET_ENDPOINT}]

OPENAPI_SCHEMA = json.dumps(openapi_schema)
print(f"Loaded OpenAPI schema: {openapi_schema['info']['title']} v{openapi_schema['info']['version']}")
print(f"Server URL: {TARGET_ENDPOINT}")
print(f"Endpoints: {list(openapi_schema['paths'].keys())}")
print(f"routingDomain: {PROXY_ALB_DNS}")

In [ ]:
response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="self-signed-proxy",
    description="Backend API with self-signed cert, routed through public cert proxy ALB via managed VPC egress",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "inlinePayload": OPENAPI_SCHEMA,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": CRED_PROVIDER_ARN,
                    "credentialParameterName": "x-api-key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
    privateEndpoint={
        "managedVpcResource": {
            "vpcIdentifier": VPC_USW2_ID,
            "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
            "endpointIpAddressType": "IPV4",
            "securityGroupIds": [PROXY_ALB_SG_ID],
            "routingDomain": PROXY_ALB_DNS,
        }
    },
)

TARGET_ID = response["targetId"]
print(f"\nTarget ID: {TARGET_ID}")
print(f"Status:    {response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nTarget is active!")
        print(f"  Managed resources: {target.get('privateEndpoint', {})}")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

## 6단계: AgentCore Gateway를 통해 API 호출

Cognito에서 액세스 토큰을 가져온 다음 게이트웨이를 통해 API 작업을 MCP 도구로 호출합니다. 트래픽 흐름은 다음과 같습니다.

```
Your request → AgentCore Gateway → VPC Lattice → Proxy ALB (public cert, TLS termination
  + host header transform) → Your API (self-signed cert, HTTPS:443)
```

In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]
print(f"Access token obtained (expires in {token_response['expires_in']}s)")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

# 사용 가능한 도구 목록 표시(MCP 도구로 노출된 API 작업)
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
print("Available tools:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# 상태 확인
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "self-signed-proxy___healthCheck", "arguments": {}},
        "id": 2,
    },
)
print("Health check:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# 항목 생성
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "self-signed-proxy___createItem",
            "arguments": {"name": "Widget", "price": 9.99},
        },
        "id": 3,
    },
)
print("Create item:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# 항목 목록 표시
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "self-signed-proxy___listItems", "arguments": {}},
        "id": 4,
    },
)
print("Items:")
print(json.dumps(response.json(), indent=2))

## 정리

1. 게이트웨이 대상과 자격 증명 공급자를 삭제합니다.
2. 프록시 ALB 리소스(boto3를 통해 생성)를 삭제합니다.
3. CDK 스택(백엔드)을 제거합니다.
4. 유지된 NLB 보안 그룹을 삭제합니다.

> **참고:** AgentCore의 관리형 Resource Gateway ENI가 NLB 보안 그룹을 계속 참조할 수 있으므로 스택을 삭제할 때 이 보안 그룹은 유지됩니다. 게이트웨이 대상이 완전히 제거된 후 수동으로 삭제하세요.

In [ ]:
# # 1단계: 게이트웨이 대상 삭제
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# print(f"Deleting target: {TARGET_ID}")
# while True:
#     try:
#         t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
#         print(f"  Status: {t['status']}")
#         time.sleep(15)
#     except agentcore.exceptions.ResourceNotFoundException:
#         print("  Target deleted.")
#         break

# # 자격 증명 공급자 삭제
# agentcore.delete_api_key_credential_provider(name="self-signed-proxy-api-key")
# print("Deleted credential provider: self-signed-proxy-api-key")

In [ ]:
# # 2단계: 프록시 ALB 리소스 삭제(boto3를 통해 생성)
# elbv2.delete_listener(ListenerArn=LISTENER_ARN)
# print("Deleted listener")
#
# elbv2.delete_load_balancer(LoadBalancerArn=PROXY_ALB_ARN)
# print("Deleting ALB...")
# waiter = elbv2.get_waiter("load_balancers_deleted")
# waiter.wait(LoadBalancerArns=[PROXY_ALB_ARN])
# print("ALB deleted")
#
# elbv2.delete_target_group(TargetGroupArn=TARGET_GROUP_ARN)
# print("Deleted target group")

In [ ]:
# # 3단계: CDK 스택 제거
# !cdk destroy SelfSignedBackend --profile {ACCOUNT_A_PROFILE} --force

In [ ]:
# # 3.1단계: 프록시 ALB SG 삭제(ENI가 계속 참조하는 경우 DependencyViolation이 발생할 수 있음)
# try:
#     ec2_client.delete_security_group(GroupId=PROXY_ALB_SG_ID)
#     print(f"Deleted proxy ALB security group: {PROXY_ALB_SG_ID}")
# except ec2_client.exceptions.ClientError as e:
#     if "DependencyViolation" in str(e):
#         print(f"SG {PROXY_ALB_SG_ID} still has dependencies. Wait a few minutes and retry.")
#     else:
#         raise

In [ ]:
# # 4단계: 유지된 NLB 보안 그룹 삭제
# # "DependencyViolation"으로 실패하면 ENI가 해제될 때까지 몇 분간 대기
# ec2_client = session.client("ec2")
# try:
#     ec2_client.delete_security_group(GroupId=NLB_SG_ID)
#     print(f"Deleted security group: {NLB_SG_ID}")
# except ec2_client.exceptions.ClientError as e:
#     if "DependencyViolation" in str(e):
#         print(f"SG {NLB_SG_ID} still has dependencies. Wait a few minutes and retry.")
#     else:
#         raise